# Reprodução das tabelas e figuras, sem retreinar

Este notebook parte dos **pesos treinados** e refaz a inferência em validação e teste, as tabelas, os testes estatísticos e as figuras de explicabilidade. Nada é treinado.

**Entradas**
1. Imagens convertidas e `splits.csv` (ver a seção *Dados* do README).
2. Pacote de pesos: uma pasta por execução, com `weights.pt` (ajuste fino) ou `head.pt` (extrator congelado) e o `run.json`. Os baselines do TP1 vêm com o `preds.npz`, porque o Random Forest não é salvo.
3. `ResNet50.pt` do RadImageNet, usado só pelo RadImageNet congelado.

**Ambiente:** GPU recomendada. No Kaggle, ligue a internet para baixar os backbones pré-treinados do `timm`.

In [ ]:
import os
if not os.path.isdir("src"):
    if not os.path.isdir("ap2-topicos"):
        !git clone -q https://github.com/arisioandradee/ap2-topicos.git
    %cd ap2-topicos
!pip install -q grad-cam captum

## 1. Caminhos
Ajuste os caminhos abaixo.

In [ ]:
IMG_DIR = "/kaggle/input/ich-img192-all/ich192"    # imagens (.bin + index_shard*.csv)
SPLITS = IMG_DIR + "/splits.csv"                    # partição por paciente
PESOS = "/kaggle/input/ich-tp2-pesos"               # pacote de pesos
RADIMAGENET = "/kaggle/input/radimagenet/ResNet50.pt"

RUNS, TABELAS, XAI, FIGURAS = "saida/runs", "saida/tables", "saida/xai", "saida/figures"

## 2. Inferência a partir dos pesos

In [ ]:
!python -m src.infer --runs "{PESOS}" --splits "{SPLITS}" --img_dir "{IMG_DIR}" --radimagenet_weights "{RADIMAGENET}" --out {RUNS}

## 3. Tabelas, intervalos de confiança e testes pareados
Leva cerca de 30 minutos em CPU por causa do bootstrap por paciente.

In [ ]:
!python -m src.make_report --splits "{SPLITS}" --runs {RUNS} --out {TABELAS}

## 4. Explicabilidade
Os casos são os de `configs/xai_cases.json`, fixados antes da geração dos mapas.

In [ ]:
!python -m src.run_xai explain --cases configs/xai_cases.json --search "{PESOS}" --img_dir "{IMG_DIR}" --out {XAI}

## 5. Figura e tabelas resumidas

In [ ]:
!python -m src.figuras --tables {TABELAS} --xai_tables {XAI} --xai_figs {XAI} --out {FIGURAS}

from IPython.display import Image, Markdown, display
display(Image(FIGURAS + "/fig_xai.png", width=900))
display(Markdown(open(FIGURAS + "/tabelas.md", encoding="utf-8").read()))

## 6. Conferência com os valores publicados
Os pesos do ajuste fino são guardados em meia precisão, então pequenas diferenças nas últimas casas são esperadas.

In [ ]:
import pandas as pd

publicado = pd.read_csv("results/tables/main_table.csv")[["config", "auc_roc_mean"]]
novo = pd.read_csv(TABELAS + "/main_table.csv")[["config", "auc_roc_mean"]]
c = publicado.merge(novo, on="config", suffixes=("_publicado", "_reproduzido"))
c["diferenca"] = c.auc_roc_mean_reproduzido - c.auc_roc_mean_publicado
display(c.round(5))
print("maior diferença absoluta:", round(c.diferenca.abs().max(), 5))